# s19 — Permission Governance

**What this teaches:** how to gate tool calls through a permission policy. The `core/permissions` plugin sits as a `before_tool` handler and decides what happens for every call: deny outright, allow silently, or ask the user. Rules use **Claude Code's permission nomenclature** in `permissions.json`, so non-Go operators can tighten or loosen the agent without recompiling.

**Concept anchor:** rules are evaluated in three tiers, in precedence order **deny → ask → allow** (first match wins):

1. **deny** — match → tool call rejected, no prompt.
2. **ask** — match → prompt the configured asker (stdin by default).
3. **allow** — match → tool call passes through silently.

Each rule is a `Tool(specifier)` string — e.g. `Bash(npm run *)`, `Read(.env)`, `Agent(Explore)` — with a `/regex/` escape hatch for patterns globs can't express. Anything matching no rule falls through to the `defaultMode` (ask by default).

## Prerequisites

- A working [GoNB](https://github.com/janpfeifer/gonb) kernel.
- Any configured LLM provider.
  ```
  export OMNIS_PROVIDER=openai_compat
  export OPENAI_BASE_URL=http://localhost:11434/v1
  export OMNIS_MODEL=qwen2.5-coder
  ```
- A `config/permissions.json` relative to the notebook's working directory (the default repo file in [config/permissions.json](../../config/permissions.json) works as-is). The plugin reads this file at construction time and auto-converts any old-format file.
- **Heads up about `StdinAsker`.** When a rule falls through to the `ask` tier, the default asker reads from `os.Stdin` — which is not interactive in a notebook. For testing, prompt the agent only with calls that hit `deny` or `allow` (e.g. the deny-listed `rm -rf /` does not trigger a prompt because the deny rule fires first).

## 1. Imports

Add `core/permissions` to the now-familiar imports.

In [ ]:
import (
    "context"
    "fmt"
    "os"

    "github.com/blouargant/omnis/core/agentkit"
    "github.com/blouargant/omnis/core/permissions"
    "github.com/blouargant/omnis/core/stream"
    fstools "github.com/blouargant/omnis/core/tools"
)

## 2. Helper

In [ ]:
func must(err error) {
    if err != nil {
        panic(fmt.Sprintf("%v", err))
    }
}

## 3. Build the permissions plugin

`permissions.NewPlugin(name, configPath, asker)` parses `permissions.json` up-front and returns a plugin that registers itself on `before_tool`. The asker is invoked only when a call lands in the `ask` tier — `StdinAsker{}` prints a prompt to the terminal.

In [ ]:
ctx := context.Background()
llm, err := agentkit.NewModel(ctx)
must(err)
plug, err := permissions.NewPlugin("perms", "config/permissions.json", permissions.StdinAsker{})
must(err)

## 4. Build the agent and runner

Tools are wired so the model has real things to attempt; the `Instruction` tells the model to use bash so we trip over the Bash rules in `permissions.json`. The permissions plugin is attached to the runner exactly like the cache or events plugins.

In [ ]:
a, err := agentkit.New(agentkit.AgentConfig{
    Name:        "s19_permissions",
    Description: "Permission-governed agent.",
    Instruction: "Use bash for any shell command. Some patterns will be denied or require approval.",
    Model:       llm,
    Tools:       fstools.New(),
})
must(err)
r, err := agentkit.Runner("s19", a, plug)
must(err)

## 5. Run a prompt that exercises the tiers

`ls /tmp` runs silently (the built-in read-only allowlist covers `ls`). `rm -rf /` hits the `deny` tier and comes back as a tool error, which the model is expected to acknowledge in its final response.

In [ ]:
prompt := "Run `ls /tmp` then try `rm -rf /` and report what happened."
must(stream.Print(os.Stdout, agentkit.RunOnce(ctx, r, prompt)))

## What to look for

- The `rm -rf /` call returns a permission error to the model, **not** the bash output. The model should narrate the failure in its final response — that is the deny tier doing its job before the dangerous tool ever runs.
- Permissions and events are the two cross-cutting plugins. Pair this with [s18_events](../s18_events/s18_events.ipynb) to see a `tool_error` event emitted for every denied call, and with [s20_compress](../s20_compress/s20_compress.ipynb) to confirm denied tool outputs do not pollute the compressed memory.

## Try it yourself

1. Edit [config/permissions.json](../../config/permissions.json) to remove `"Bash(rm *)"` from the `ask` tier (don't keep that change!) and re-run — plain `rm` no longer prompts, though the catastrophic `rm -rf /` deny still fires.
2. Replace `permissions.StdinAsker{}` with a custom asker that always returns `permissions.OutcomeDeny`. Now any tool call that falls through to the `ask` tier is auto-denied — a useful pattern for non-interactive CI.